In [1]:
import numpy as np

class GRU:
    def __init__(self, input_size, hidden_size, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        
        # ==================== 初始化参数 ====================
        # 使用 Xavier 初始化来保持梯度的稳定性
        std = 1.0 / np.sqrt(hidden_size)
        
        # 1. 更新门 (z) 参数
        self.Wz = np.random.uniform(-std, std, (input_size, hidden_size))
        self.Uz = np.random.uniform(-std, std, (hidden_size, hidden_size))
        self.bz = np.zeros((1, hidden_size))
        
        # 2. 重置门 (r) 参数
        self.Wr = np.random.uniform(-std, std, (input_size, hidden_size))
        self.Ur = np.random.uniform(-std, std, (hidden_size, hidden_size))
        self.br = np.zeros((1, hidden_size))
        
        # 3. 候选状态 (h_tilde) 参数
        self.Wh = np.random.uniform(-std, std, (input_size, hidden_size))
        self.Uh = np.random.uniform(-std, std, (hidden_size, hidden_size))
        self.bh = np.zeros((1, hidden_size))
        
        # 用于参数更新的梯度缓存
        self.reset_grads()

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def d_sigmoid(self, sigmoid_out):
        # Sigmoid 导数: y * (1 - y)
        return sigmoid_out * (1 - sigmoid_out)

    def tanh(self, x):
        return np.tanh(x)
    
    def d_tanh(self, tanh_out):
        # Tanh 导数: 1 - y^2
        return 1 - tanh_out ** 2

    def reset_grads(self):
        """每轮反向传播前清空梯度"""
        self.dWz, self.dUz, self.dbz = np.zeros_like(self.Wz), np.zeros_like(self.Uz), np.zeros_like(self.bz)
        self.dWr, self.dUr, self.dbr = np.zeros_like(self.Wr), np.zeros_like(self.Ur), np.zeros_like(self.br)
        self.dWh, self.dUh, self.dbh = np.zeros_like(self.Wh), np.zeros_like(self.Uh), np.zeros_like(self.bh)

    def forward(self, inputs, h_prev_init=None):
        """
        处理整个序列的前向传播
        inputs: (seq_len, batch_size, input_size)
        """
        seq_len, batch_size, _ = inputs.shape
        
        # 如果没提供初始隐藏状态，全0初始化
        if h_prev_init is None:
            h_prev = np.zeros((batch_size, self.hidden_size))
        else:
            h_prev = h_prev_init

        # 缓存每个时刻的状态，用于反向传播
        # cache 结构: 字典列表，每个元素对应一个时间步 t
        self.cache = [] 
        
        h_t = h_prev
        hs = {} # 存储所有时刻的隐藏状态输出
        hs[-1] = h_t
        
        for t in range(seq_len):
            x_t = inputs[t]
            
            # --- 1. 重置门 r_t ---
            r_t = self.sigmoid(np.dot(x_t, self.Wr) + np.dot(h_t, self.Ur) + self.br)
            
            # --- 2. 更新门 z_t ---
            z_t = self.sigmoid(np.dot(x_t, self.Wz) + np.dot(h_t, self.Uz) + self.bz)
            
            # --- 3. 候选状态 h_tilde ---
            # 重置门作用于旧状态
            h_reset = r_t * h_t
            h_tilde = self.tanh(np.dot(x_t, self.Wh) + np.dot(h_reset, self.Uh) + self.bh)
            
            # --- 4. 最终状态 h_t ---
            h_prev_t = h_t # 保存更新前的状态
            h_t = (1 - z_t) * h_prev_t + z_t * h_tilde
            
            # 保存结果和中间变量
            hs[t] = h_t
            self.cache.append({
                'x_t': x_t,
                'h_prev': h_prev_t, # t-1 时刻的 h
                'r_t': r_t,
                'z_t': z_t,
                'h_tilde': h_tilde,
                'h_reset': h_reset  # r_t * h_prev
            })
            
        return hs, h_t

    def backward(self, dh_from_next_layer):
        """
        反向传播 (BPTT)
        dh_from_next_layer: 来自上层（比如输出层）的梯度序列
                            字典格式 {t: gradient} 或者 数组
        """
        seq_len = len(self.cache)
        
        # 初始化流向“未来”的梯度 (最后一个时刻之后没有梯度传回来)
        dh_next = np.zeros((1, self.hidden_size)) 
        
        self.reset_grads()

        # === 时间反向循环 (T, T-1, ..., 0) ===
        for t in reversed(range(seq_len)):
            # 获取当前时刻的缓存
            vars = self.cache[t]
            x_t = vars['x_t']
            h_prev = vars['h_prev'] # h_{t-1}
            r_t = vars['r_t']
            z_t = vars['z_t']
            h_tilde = vars['h_tilde']
            h_reset = vars['h_reset']
            
            # 1. 汇总梯度：来自下一时刻的梯度 + 来自当前时刻输出层的梯度
            # 假设 dh_from_next_layer 是个字典，存了每个时刻的 loss 对 h_t 的导数
            dh_t = dh_from_next_layer[t] + dh_next
            
            # ===============================================
            #  对公式 h_t = (1 - z) * h_prev + z * h_tilde 求导
            # ===============================================
            
            # 对 z_t 求导 (经过 h_t)
            dz_t = dh_t * (h_tilde - h_prev) * self.d_sigmoid(z_t)
            
            # 对 h_tilde 求导 (经过 h_t)
            dh_tilde = dh_t * z_t * self.d_tanh(h_tilde)
            
            # 对 h_prev 求导 (直接路径部分，后面还要加上通过 r, z 回传的部分)
            dh_prev_running = dh_t * (1 - z_t)

            # ===============================================
            #  计算各权重矩阵的梯度
            # ===============================================
            
            # --- A. 处理 h_tilde 分支 ---
            # h_tilde = tanh(x*Wh + (r*h_prev)*Uh + bh)
            self.dWh += np.dot(x_t.T, dh_tilde)
            self.dUh += np.dot(h_reset.T, dh_tilde)
            self.dbh += np.sum(dh_tilde, axis=0, keepdims=True)
            
            # 反向传播到 h_reset = r_t * h_prev
            dh_reset = np.dot(dh_tilde, self.Uh.T)
            
            # 分解 h_reset 对 r_t 和 h_prev 的梯度
            dr_t_temp = dh_reset * h_prev
            dh_prev_running += dh_reset * r_t # 加上通过 h_tilde 回传给 h_prev 的梯度
            
            # --- B. 处理 Reset Gate (r_t) ---
            # r_t 经过 sigmoid
            dr_t = dr_t_temp * self.d_sigmoid(r_t)
            
            self.dWr += np.dot(x_t.T, dr_t)
            self.dUr += np.dot(h_prev.T, dr_t)
            self.dbr += np.sum(dr_t, axis=0, keepdims=True)
            
            # 加上通过 r_t 回传给 h_prev 的梯度
            dh_prev_running += np.dot(dr_t, self.Ur.T)
            
            # --- C. 处理 Update Gate (z_t) ---
            # 上面已经算过 dz_t 了，这里直接算权重
            self.dWz += np.dot(x_t.T, dz_t)
            self.dUz += np.dot(h_prev.T, dz_t)
            self.dbz += np.sum(dz_t, axis=0, keepdims=True)
            
            # 加上通过 z_t 回传给 h_prev 的梯度
            dh_prev_running += np.dot(dz_t, self.Uz.T)
            
            # --- 更新流向上一时刻的梯度 ---
            dh_next = dh_prev_running

        # 为了防止梯度爆炸 (Exploding Gradients)，这是 RNN 中必须的步骤
        for dparam in [self.dWz, self.dUz, self.dbz, self.dWr, self.dUr, self.dbr, self.dWh, self.dUh, self.dbh]:
            np.clip(dparam, -1, 1, out=dparam)

    def update_params(self):
        """使用简单的 SGD 更新参数"""
        for param, dparam in zip([self.Wz, self.Uz, self.bz, self.Wr, self.Ur, self.br, self.Wh, self.Uh, self.bh],
                                 [self.dWz, self.dUz, self.dbz, self.dWr, self.dUr, self.dbr, self.dWh, self.dUh, self.dbh]):
            param -= self.lr * dparam

# ==========================================
# 完整训练流程演示 (Mock Training)
# ==========================================

# 1. 设置数据
seq_len = 5
batch_size = 1
input_size = 4
hidden_size = 8

# 模拟输入序列 (时间步=5, batch=1, 特征=4)
inputs = np.random.randn(seq_len, batch_size, input_size)
# 模拟目标输出 (假设我们要让每个时刻的 h_t 接近某个随机值)
targets = np.random.randn(seq_len, batch_size, hidden_size)

# 2. 初始化模型
model = GRU(input_size, hidden_size, learning_rate=0.1)

print("=== 开始训练模拟 ===")
for epoch in range(10): # 训练 10 轮
    
    # --- 前向传播 ---
    hs, _ = model.forward(inputs)
    
    # --- 计算 Loss 和 初始梯度 ---
    loss = 0
    dh_grads = {}
    for t in range(seq_len):
        # 简单的 MSE Loss: L = 0.5 * (h_t - target)^2
        # dL/dh_t = h_t - target
        loss += 0.5 * np.sum((hs[t] - targets[t])**2)
        dh_grads[t] = hs[t] - targets[t]
        
    print(f"Epoch {epoch}, Loss: {loss:.4f}")
    
    # --- 反向传播 ---
    model.backward(dh_grads)
    
    # --- 更新参数 ---
    model.update_params()

print("=== 训练结束，Loss 应该下降 ===")

=== 开始训练模拟 ===
Epoch 0, Loss: 17.6981
Epoch 1, Loss: 15.3269
Epoch 2, Loss: 13.8828
Epoch 3, Loss: 12.9219
Epoch 4, Loss: 12.1786
Epoch 5, Loss: 11.5763
Epoch 6, Loss: 11.0689
Epoch 7, Loss: 10.6294
Epoch 8, Loss: 10.2407
Epoch 9, Loss: 9.8921
=== 训练结束，Loss 应该下降 ===
